# Atividade 07

Henrique Andrade Lopes - 105459

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt

import cv2
import numpy as np

from tqdm import trange
import tqdm
import random
import torch
import torchvision

from PIL import Image
from torchvision.models import AlexNet, AlexNet_Weights
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import vgg16, VGG16_Weights

import scipy.io

from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors

import math

# Fixa a semente para o Python e Numpy
SEED = 105459  ### TROCAR PELA MATRÌCULA
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
def show_top_images_grid(dataset_path, indices, id_test, ids, labels, cols=4):
    """
    Fetches the query image and its top matches, displaying them in a grid.
    cols: Number of columns you want in the grid.
    """
    images_to_show = []
    titles = []
    
    # 1. Fetch the Query Image
    label = (ids[id_test] - 1) // 80
    name = f"{dataset_path}/jpg/{label}/image_{str(ids[id_test]).zfill(4)}.jpg"
    
    image = cv2.imread(name)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    images_to_show.append(image)
    titles.append(f"Query Label: {labels[id_test]}\n(ID: {ids[id_test]})")

    indice = 0
    # 2. Fetch the Match Images from indices
    for i in indices[0]:
        label_i = labels[i]
        name = f"{dataset_path}/jpg/{label_i}/image_{str(ids[i]).zfill(4)}.jpg"
        
        image = cv2.imread(name)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        images_to_show.append(image)
        titles.append(f"{indice}° Match: ID {ids[i]} \n Label: {label_i} - {'Correct' if label_i == labels[id_test] else 'Incorrect'}")
        indice += 1 
    
    # Create the Figure and Subplots Header
    # Adjust figsize based on your layout preference (width, height)
    fig_header, axes_header = plt.subplots(1, 2, figsize=(20, 4))

    axes_header[0].imshow(images_to_show[0])
    axes_header[0].set_title(f'Query. \n Label: {labels[id_test]}\n(ID: {ids[id_test]})')
    axes_header[0].axis('off') # Hide the axis ticks and borders

    axes_header[1].imshow(images_to_show[1])
    axes_header[1].set_title(f'Sanity Test. \n Label: {labels[indices[0][0]]}\n(ID: {ids[indices[0][0]]})')
    axes_header[1].axis('off') # Hide the axis ticks and borders
    
    
    # 3. Calculate Grid Dimensions
    n_images = len(images_to_show) - 2
    rows = math.ceil(n_images / cols)
    
    # 4. Create the Figure and Subplots
    # Adjust figsize based on your layout preference (width, height)
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
    
    # Flatten axes array to easily iterate over it, even if it's a 2D grid
    if n_images > 1:
        axes = axes.flatten()
    else:
        axes = [axes] # Handle edge case where there's only 1 image
        
    # 5. Populate the Grid
    for idx in range(len(axes)):
        if idx < n_images:
            axes[idx].imshow(images_to_show[idx+2])
            axes[idx].set_title(titles[idx+2])
            axes[idx].axis('off') # Hide the axis ticks and borders
        else:
            # Turn off empty subplots if the grid isn't perfectly filled
            axes[idx].axis('off') 
            
    plt.tight_layout()
    plt.show()

In [ ]:
def retrieve_single_image(space, labels, dataset_path, test=True, top=10, metric='cosine'):

    knn = NearestNeighbors(n_neighbors=top+1, metric=metric).fit(space)

    mat = scipy.io.loadmat(dataset_path + '/datasplits.mat')

    if test:
        ids = mat['tst1'][0]
    else:
        ids = mat['val1'][0]

    id_test = random.randrange(len(ids))

    distances, indices = knn.kneighbors(space[id_test].reshape(1, -1))

    show_top_images_grid(dataset_path, indices, id_test, ids, labels)

    labels_top = [int(labels[i]) for i in indices[0]]

    accuracy = sum(np.equal(labels[id_test], labels_top))
    accuracy = ((accuracy - 1) / top) * 100

    print(f'Accuracy for image id {ids[id_test]}: {accuracy:5.2f}%')
    print(f'Image: {ids[id_test]} with label {labels[id_test]}')
    print(f'Closest image: {ids[indices[0][0]]} with distance {distances[0][0]} and label {labels[indices[0][0]]}')
    print('Distances: ', distances)
    print('Indices: ', indices[0])
    print('Labels: ', labels_top)

In [ ]:
def create_descriptor(image_path):
    image = Image.open(image_path).convert("RGB")
    image = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        desc = model_descriptor(image)

    desc = torch.nn.functional.normalize(desc, p=2, dim=1)

    return desc.cpu().numpy().ravel()


In [ ]:
def represent_dataset(dataset_path, test=False, algorithm='alexnet'):
    mat = scipy.io.loadmat(dataset_path + '/datasplits.mat')

    if test:
        ids = mat['tst1'][0]
    else:
        ids = mat['val1'][0]

    space = []
    labels = []

    for id in tqdm.tqdm(ids, desc='Processing test set' if test else 'Processing val set'):

        label = (id - 1) // 80
        name = dataset_path + '/jpg/' + str(label) + '/image_' + str(id).zfill(4) + '.jpg'

       
        desc = create_descriptor(name)
        
        space.append(desc)
        labels.append(label)

    space = np.array(space)
    labels = np.array(labels)

    print(' -> [I] Space Describing Info:\n',
          '\nNumber of images: ', len(space),
          '\nNumber of labels: ', len(labels),
          '\nDimension: ', len(space[0])
    )

    return space, labels

In [ ]:
def run_experiment(space, labels, dataset_path, test=False, top=10, metric='cosine'):

    knn = NearestNeighbors(n_neighbors=top+1, metric=metric).fit(space)

    mat = scipy.io.loadmat(dataset_path + '/datasplits.mat')

    if test :
        ids = mat['tst1'][0] #  'tst1' or 'trn1' or 'val1'
    else :
        ids = mat['val1'][0] #  'tst1' or 'trn1' or 'val1'

    accuracy_t = 0

    for i, id_test in enumerate(tqdm.tqdm(ids, desc='running the test phase' if test else 'running the val phase')):

        query_desc = space[i]

        indices = knn.kneighbors(query_desc.reshape(1, -1))[1]

        labels_top = [labels[j] for j in indices[0]]

        label = labels[i]

        accuracy = sum( np.equal(labels_top, label) )
        accuracy =( (accuracy-1)/(top) ) * 100
        accuracy_t = accuracy_t + accuracy

    print('Average accuracy in the', 'test' if test else 'validation', f'set: {accuracy_t/len(ids):5.2f}%')


# BOVW

In [ ]:

def create_vocabulary(dataset_path):
    mat = scipy.io.loadmat(dataset_path + '/datasplits.mat')
    ids = mat['trn1'][0] #  'val1' or 'tst1' 

    train_descs = []

    for id in tqdm.tqdm(ids, desc='Processing train set'):

        label = (id - 1) // 80
        name = dataset_path + '/jpg/' + str(label) + '/image_' + str(id).zfill(4) + '.jpg'

        desc = create_descriptor(name)
        train_descs.append(desc)

    train_descs = np.array(train_descs)

    print(' -> [I] Image Loader Info:\n',
          '\nTrain len: ', len(train_descs),
          '\nNumber of images: ', len(ids),
          '\nDescriptor size: ', len(train_descs[0])
    )

    return train_descs


def create_dictionary_kmeans(vocabulary, num_cluster):

    print(' -> [I] Dictionary Info:\n',
          '\nTrain len: ', len(vocabulary),
          '\nDimension: ', len(vocabulary[0]),
          '\nClusters: ', num_cluster
    )

    dictionary = MiniBatchKMeans(n_clusters=num_cluster, batch_size=1000, random_state=SEED)

    print('Learning dictionary by Kmeans...')
    dictionary = dictionary.fit(vocabulary)
    print('Done.')

    return dictionary

    
def create_bovw_descriptors(image_path, dictionary):

    desc = create_descriptor(image_path).reshape(1, -1)

    predicted = dictionary.predict(desc)

    desc_bovw = np.histogram(predicted, bins=range(0, dictionary.n_clusters + 1))[0]

    return desc_bovw


def represent_dataset_bovw(dataset_path, dictionary, test=False):

    mat = scipy.io.loadmat(dataset_path + '/datasplits.mat')

    if test:
        ids = mat['tst1'][0]
    else:
        ids = mat['val1'][0]

    space = []
    labels = []

    for id in tqdm.tqdm(ids, desc='Processing test set' if test else 'Processing val set'):

        label = (id - 1) // 80
        name = dataset_path + '/jpg/' + str(label) + '/image_' + str(id).zfill(4) + '.jpg'

        desc_bovw = create_bovw_descriptors(name, dictionary)

        space.append(desc_bovw)
        labels.append(label)

    space = np.array(space)
    labels = np.array(labels)

    print(' -> [I] Space Describing Info:\n',
          '\nNumber of images: ', len(space),
          '\nNumber of labels: ', len(labels),
          '\nDimension: ', len(space[0])
    )

    return space, labels

# Alex Net


In [ ]:
#Codigo da aula pratica deep Descriptor 
class AlexNetDescriptor(AlexNet):
    def __init__(self):
        super(self.__class__, self).__init__()
        
        self.classifier = torch.nn.Sequential(
            torch.nn.Dropout(),
            torch.nn.Linear(256 * 6 * 6, 4096),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(),
            torch.nn.Linear(4096, 4096),
            # torch.nn.ReLU(inplace=True),
            # torch.nn.Linear(4096, 1000),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model_descriptor = AlexNetDescriptor()

# Directly load weights (it is essential to use the strict equals to False, otherwise it will not work).
model_descriptor.load_state_dict(torchvision.models.alexnet(weights=AlexNet_Weights.IMAGENET1K_V1).state_dict(), strict=False)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = AlexNet_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

model_descriptor = model_descriptor.to(device)
model_descriptor.eval()

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 128

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=False, metric='euclidean')

In [ ]:
retrieve_single_image(space, labels, dataset_path, test=True)

# Resnet

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

weights = ResNet50_Weights.DEFAULT
preprocess = weights.transforms()

model_descriptor = resnet50(weights=weights)

# Remove a camada classificadora final.
model_descriptor.fc = torch.nn.Identity()

model_descriptor = model_descriptor.to(device)
model_descriptor.eval()

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 300

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=False, metric='euclidean')

In [ ]:
retrieve_single_image(space, labels, dataset_path, test=True)

# VGG

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

weights = VGG16_Weights.DEFAULT
preprocess = weights.transforms()

model_descriptor = vgg16(weights=weights)

# Remove a camada classificadora final.
model_descriptor.classifier[-1] = torch.nn.Identity()

model_descriptor = model_descriptor.to(device)
model_descriptor.eval()

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 128

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=False, metric='euclidean')

In [ ]:
retrieve_single_image(space, labels, dataset_path, test=True)

# Testando todos os casos


## AlexNet

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = AlexNet_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

model_descriptor = model_descriptor.to(device)
_ = model_descriptor.eval()


### BoVW Mudando Dic. Size

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 50

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 100

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 200

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 400

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

### Sem utilizar BoVW

In [ ]:
dataset_path = r'E:\Inf692'
space, labels = represent_dataset(dataset_path, test=False)
run_experiment(space, labels, dataset_path, test=False)

## ResNet

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

weights = ResNet50_Weights.DEFAULT
preprocess = weights.transforms()

model_descriptor = resnet50(weights=weights)

# Remove a camada classificadora final.
model_descriptor.fc = torch.nn.Identity()

model_descriptor = model_descriptor.to(device)
_ = model_descriptor.eval()

### BoVW Mudando Dic. Size

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 50

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 100

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 200

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 400

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

### Sem utilizar BoVW

In [ ]:
dataset_path = r'E:\Inf692'
space, labels = represent_dataset(dataset_path, test=False)
run_experiment(space, labels, dataset_path, test=False)

## VGG16

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

weights = VGG16_Weights.DEFAULT
preprocess = weights.transforms()

model_descriptor = vgg16(weights=weights)

# Remove a camada classificadora final.
model_descriptor.classifier[-1] = torch.nn.Identity()

model_descriptor = model_descriptor.to(device)
_ = model_descriptor.eval()

### BoVW Mudando Dic. Size

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 50

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 100

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 200

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

In [ ]:
dataset_path = r'E:\Inf692'
num_cluster = 400

vocabulary = create_vocabulary(dataset_path)
dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster)

space, labels = represent_dataset_bovw(dataset_path, dictionary, test=False)

run_experiment(space, labels, dataset_path, test=True, metric='euclidean')

### Sem utilizar BoVW

In [ ]:
dataset_path = r'E:\Inf692'
space, labels = represent_dataset(dataset_path, test=False)
run_experiment(space, labels, dataset_path, test=False)

## Comparação dos modelos

| Modelo   | Configuração | Dimensão do descritor | Acurácia no teste |
| -------- | ------------ | --------------------: | ----------------: |
| AlexNet  | BoVW K=50    |                    50 |            44,35% |
| AlexNet  | BoVW K=100   |                   100 |            43,47% |
| AlexNet  | BoVW K=200   |                   200 |            36,26% |
| AlexNet  | BoVW K=400   |                   400 |            25,26% |
| AlexNet  | Sem BoVW     |                  4096 |            62,47% |
| ResNet50 | BoVW K=50    |                    50 |            51,71% |
| ResNet50 | BoVW K=100   |                   100 |            42,88% |
| ResNet50 | BoVW K=200   |                   200 |            39,82% |
| ResNet50 | BoVW K=400   |                   400 |            24,26% |
| ResNet50 | Sem BoVW     |                  2048 |        **63,15%** |
| VGG16    | BoVW K=50    |                    50 |            44,35% |
| VGG16    | BoVW K=100   |                   100 |            43,47% |
| VGG16    | BoVW K=200   |                   200 |            36,26% |
| VGG16    | BoVW K=400   |                   400 |            25,26% |
| VGG16    | Sem BoVW     |                  4096 |            59,62% |




## Discussao

O melhor resultado geral foi obtido pela ResNet50 sem utilização do BoVW. Quando consideramos apenas os métodos que utilizam BoVW, a ResNet50 também apresentou o melhor desempenho, com BoVW = 50.

Ao comparar esses resultados com os da atividade03, que utilizava SIFT, ORB e RANDOM, nota-se um ganho considerável de acurácia, já que mesmo os piores valores deste teste foram superiores aos melhores resultados da atividade anterior.

Outra comparação importante está relacionada ao tamanho K do BoVW. Enquanto na atividade03 um K = 200 foi considerado o ideal, nesta atividade valores menores de K apresentaram melhores resultados, especialmente K = 50. Isso indica que, para descritores profundos, um vocabulário menor foi suficiente e acabou gerando uma representação mais eficiente.